In [125]:
using LinearAlgebra
using Random
using ProgressMeter
using JLD2
using Ket
using MosekTools
using ppt2

In [126]:
rng = Xoshiro(0)

Xoshiro(0xdb2fa90498613fdf, 0x48d73dc42d195740, 0x8c49bc52dc8a77ea, 0x1911b814c02405e8, 0x22a21880af5dc689)

In [ ]:
function rand_vec(dims...; rng=Random.GLOBAL_RNG)
    return float(rand(rng, -1:1, dims...))
end

rand_vec (generic function with 1 method)

In [232]:
function rand_ppt(n::Int, m::Int; rng=Random.GLOBAL_RNG, rand_vec=_rand_vec, symmetrize=true)
    A = rand_vec(n*m, n*m; rng=rng)
    rho = A * A'
    if symmetrize
        for i in 1:m, j in i+1:m
            rows = (i - 1) * n + 1:i * n
            cols = (j - 1) * n + 1:j * n
            sym = (rho[rows, cols] + rho[cols, rows]) / 2
            rho[rows, cols] = sym
            rho[cols, rows] = sym
        end
    end
    delta = eigmin(partial_transpose(rho, 1))
    if delta < 0
        return rho - 1.5 * delta * I
    end
    return rho
end

rand_ppt (generic function with 1 method)

In [ ]:
symmetrize = false
ppt = rand_ppt(3, 3; rng, rand_vec, symmetrize)

9×9 Matrix{Float64}:
  50.2214  -10.0       6.0       6.0     …  -17.0     -16.0       9.0
 -10.0      65.2214   11.0     -10.0        -10.0     -11.0     -38.0
   6.0      11.0      64.2214   19.0        -16.0      -7.0     -13.0
   6.0     -10.0      19.0      70.2214      -8.0      14.0       3.0
   5.0       4.0      11.0     -10.0         -3.0     -11.0     -11.0
  17.0      18.0      41.0      19.0     …  -21.0     -15.0     -17.0
 -17.0     -10.0     -16.0      -8.0         65.2214    6.0       0.0
 -16.0     -11.0      -7.0      14.0          6.0      46.2214    3.0
   9.0     -38.0     -13.0       3.0          0.0       3.0      69.2214

In [241]:
r, w = entanglement_robustness(ppt, [3, 3], 2; solver=Mosek.Optimizer)
println("Robustness: ", r)
w

9×9 Symmetric{Float64, Matrix{Float64}}:
  0.0389838    0.0171671    0.133075    …  -0.00472515  -0.0366283
  0.0171671    0.00755979   0.0586017       0.0515358    0.399494
  0.133075     0.0586017    0.454267        0.0021405    0.0165927
 -0.0163544   -0.0177998   -0.0710019       0.0048993    0.0195429
 -0.0072019   -0.00783841  -0.0312667      -0.0534352   -0.213148
 -0.0558275   -0.0607616   -0.242373    …  -0.00221939  -0.00885296
 -0.0107301    0.11703      0.00486074     -0.0322118   -0.00133789
 -0.00472515   0.0515358    0.0021405       0.351324     0.014592
 -0.0366283    0.399494     0.0165927       0.014592     0.000606068

In [17]:
eigvals(rand_ppt(3, 3; rng, rand_vec))

9-element Vector{Float64}:
 -6.106227258315638e-16
  0.27747906604368605
  0.6784479339461043
  1.0892311764125195
  1.1234898018587336
  1.4009688679024193
  1.5244586697611526
  1.8019377358048376
  6.212706559392318

In [229]:
function gen_ppt_ent_dps(n::Int, m::Int; n_trials::Int=10000, tol::Float64=1e-6, rng=Random.GLOBAL_RNG, rand_vec=rand_vec)
    @showprogress for i in 1:n_trials
        ppt = rand_ppt(n, m; rng, rand_vec)
        r, w = entanglement_robustness(ppt, [n, m], 2; solver=Mosek.Optimizer)
        if r > tol
            println("DPS detects entanglement: $r")
            return ppt, w
        end
    end
    return nothing, nothing
end

function gen_ppt_ent(n::Int, m::Int, forms; n_trials::Int=10000, tol::Float64=1e-6, rng=Random.GLOBAL_RNG, rand_vec=rand_vec)
    @showprogress for i in 1:n_trials
        ppt = rand_ppt(n, m; rng, rand_vec)
        r, idx = findmin(dot.(forms, Ref(ppt)))
        if r < -tol
            println("Witness $idx detects entanglement: $r")
            return ppt, forms[idx]
        end
    end
    return nothing, nothing
end

gen_ppt_ent (generic function with 1 method)

In [230]:
gen_ppt_ent_dps(3, 3)

Progress:   7%|███                                      |  ETA: 0:10:05┌ Warning: Mosek.MSK_RES_TRM_STALL
└ @ Ket /Users/noah/.julia/packages/Ket/Qqqjl/src/entanglement.jl:256
Progress:   9%|███▊                                     |  ETA: 0:09:57┌ Warning: Mosek.MSK_RES_TRM_STALL
└ @ Ket /Users/noah/.julia/packages/Ket/Qqqjl/src/entanglement.jl:256
Progress:  14%|█████▋                                   |  ETA: 0:09:22┌ Warning: Mosek.MSK_RES_TRM_STALL
└ @ Ket /Users/noah/.julia/packages/Ket/Qqqjl/src/entanglement.jl:256
Progress:  16%|██████▋                                  |  ETA: 0:09:06┌ Warning: Mosek.MSK_RES_TRM_STALL
└ @ Ket /Users/noah/.julia/packages/Ket/Qqqjl/src/entanglement.jl:256
Progress:  16%|██████▋                                  |  ETA: 0:09:05┌ Warning: Mosek.MSK_RES_TRM_STALL
└ @ Ket /Users/noah/.julia/packages/Ket/Qqqjl/src/entanglement.jl:256
Progress:  16%|██████▋                                  |  ETA: 0:09:05┌ Warning: Mosek.MSK_RES_TRM_STALL
└ @ Ket /Users

(nothing, nothing)

In [19]:
forms = jldopen("../pncp_forms_4x4.jld2") do file
    vcat([file[k] for k in keys(file)]...)
end;

In [27]:
ppt, wit = gen_ppt_ent(4, 4, forms; n_trials=1000000)

Progress:  74%|██████████████████████████████▍          |  ETA: 0:02:40Excessive output truncated after 524319 bytes.

(nothing, nothing)

In [ ]:
open("ppt_entangled.txt","a") do io
    println(io, "PPT:")
    println(io, ppt)
    println(io, "WIT:")
    println(io, wit)
end

In [1]:
ppt = [8.324714883327928 5.395499999999999 4.991899999999999 5.0106 3.4883 4.4611 4.04045 4.41475 4.2971 5.205550000000001 4.589700000000001 5.0504999999999995 4.6375 5.577699999999999 4.5161999999999995 4.6313; 5.395499999999999 8.818614883327928 4.807300000000001 5.3617 4.4611 5.3919 4.8859 4.8402 5.205550000000001 5.4773000000000005 4.77595 4.607150000000001 5.577699999999999 6.1588 5.046749999999999 4.912050000000001; 4.991899999999999 4.807300000000001 6.91711488332793 4.8252999999999995 4.04045 4.8859 4.9305 5.108750000000001 4.589700000000001 4.77595 4.235900000000001 4.515999999999999 4.5161999999999995 5.046749999999999 4.0889 3.85935; 5.0106 5.3617 4.8252999999999995 7.81731488332793 4.41475 4.8402 5.108750000000001 4.8759 5.0504999999999995 4.607150000000001 4.515999999999999 4.8785 4.6313 4.912050000000001 3.85935 3.9306999999999994; 3.4883 4.4611 4.04045 4.41475 5.62981488332793 4.3037 3.8934 4.0627 3.4364999999999997 3.7835 4.7585 4.09445 2.7671 4.0016 4.4462 4.197850000000001; 4.4611 5.3919 4.8859 4.8402 4.3037 8.313414883327928 4.6066 5.324599999999999 3.7835 4.393 5.12075 4.65905 4.0016 5.5059 5.45425 5.222049999999999; 4.04045 4.8859 4.9305 5.108750000000001 3.8934 4.6066 9.375014883327928 5.6581 4.7585 5.12075 4.5592 5.14645 4.4462 5.45425 5.0594 4.742900000000001; 4.41475 4.8402 5.108750000000001 4.8759 4.0627 5.324599999999999 5.6581 8.474314883327928 4.09445 4.65905 5.14645 5.500400000000001 4.197850000000001 5.222049999999999 4.742900000000001 4.4424; 4.2971 5.205550000000001 4.589700000000001 5.0504999999999995 3.4364999999999997 3.7835 4.7585 4.09445 8.195114883327928 4.351499999999999 4.9598 4.416 4.3749 4.6879 4.259600000000001 4.1771; 5.205550000000001 5.4773000000000005 4.77595 4.607150000000001 3.7835 4.393 5.12075 4.65905 4.351499999999999 7.429614883327929 4.6678 4.0280000000000005 4.6879 5.330900000000001 4.9978 4.43675; 4.589700000000001 4.77595 4.235900000000001 4.515999999999999 4.7585 5.12075 4.5592 5.14645 4.9598 4.6678 8.176814883327928 4.5202 4.259600000000001 4.9978 4.535 4.75335; 5.0504999999999995 4.607150000000001 4.515999999999999 4.8785 4.09445 4.65905 5.14645 5.500400000000001 4.416 4.0280000000000005 4.5202 8.089914883327928 4.1771 4.43675 4.75335 3.6333; 4.6375 5.577699999999999 4.5161999999999995 4.6313 2.7671 4.0016 4.4462 4.197850000000001 4.3749 4.6879 4.259600000000001 4.1771 7.207314883327928 4.8236 4.1307 3.534; 5.577699999999999 6.1588 5.046749999999999 4.912050000000001 4.0016 5.5059 5.45425 5.222049999999999 4.6879 5.330900000000001 4.9978 4.43675 4.8236 9.01301488332793 4.9654 5.2963; 4.5161999999999995 5.046749999999999 4.0889 3.85935 4.4462 5.45425 5.0594 4.742900000000001 4.259600000000001 4.9978 4.535 4.75335 4.1307 4.9654 6.700314883327929 3.6059; 4.6313 4.912050000000001 3.85935 3.9306999999999994 4.197850000000001 5.222049999999999 4.742900000000001 4.4424 4.1771 4.43675 4.75335 3.6333 3.534 5.2963 3.6059 6.7057148833279285]
wit = [0.04117417647075984 -0.016537396283640763 -0.05593503423036073 -0.054512025987739354 0.09696316653755124 0.01146907076164274 -0.05152856856475329 -0.0564670902166254 0.03968792396878599 0.004486493947602797 -0.04212299488121628 -0.032642620569597956 0.02865266515300746 -0.0015043462202460841 -0.06685464101272384 -0.05657730989307746; -0.016537396283640763 0.00664225971831646 0.022466067107733974 0.021894511139636398 0.011469072782594786 -0.024855094528073503 -0.047791062204041536 -0.044065197296861446 0.004486494767484458 -0.010006435105274674 -0.010831523119193389 -0.013933324280692464 -0.0015043458158730987 -0.003413852320466667 0.013261579227509238 0.009479460973031458; -0.05593503423036073 0.022466067107733974 0.07598769051457849 0.07405449311491226 -0.05152856534843607 -0.04779106073238241 -0.038944343609453064 -0.02946378360851877 -0.04212299439943243 -0.010831522194109528 0.041203264273983756 0.028731743372867102 -0.0668546421287193 0.013261580222624721 0.12876482598366956 0.11383779262999263; -0.054512025987739354 0.021894511139636398 0.07405449311491226 0.07217054423358085 -0.0564670873429001 -0.044065195765660835 -0.029463783264250816 -0.020440372497804237 -0.03264261977732932 -0.013933323510621997 0.028731742931381858 0.01686811409449618 -0.05657731063379868 0.009479461803078005 0.1138377921610535 0.099586916927369; 0.09696316653755124 0.011469072782594786 -0.05152856534843607 -0.0564670873429001 0.22834392233553044 0.14573177309197421 0.06750986351431859 0.03635827509666396 0.09346324145116293 0.059159784757594355 -0.0218967061719174 -0.007561186520363213 0.06747578507343358 0.03154000499124446 -0.10163212111738537 -0.08319810092528214; 0.01146907076164274 -0.024855094528073503 -0.04779106073238241 -0.044065195765660835 0.14573177309197421 0.09300781022031857 0.043085634860916475 0.023204283571460704 0.059159784737169076 0.03744404394686794 -0.014119472861164016 -0.004903594483251425 0.03154000452326787 0.01277455744466305 -0.06826988156902396 -0.05493294001185737; -0.05152856856475329 -0.047791062204041536 -0.038944343609453064 -0.029463783264250816 0.06750986351431859 0.043085634860916475 0.019959344559798056 0.010749342523075129 -0.02189670812727101 -0.014119474114086638 -0.021117009463502447 -0.010121793952609958 -0.10163212597069286 -0.06826988455541702 -0.0659931177562143 -0.04395646974962552; -0.0564670902166254 -0.044065197296861446 -0.02946378360851877 -0.020440372497804237 0.03635827509666396 0.023204283571460704 0.010749342523075129 0.005789215380174019 -0.007561187422134356 -0.00490359506067532 -0.010121793911433849 -0.004777458937143693 -0.08319810466268614 -0.054932942344648426 -0.043956470091493345 -0.028205329715387483; 0.03968792396878599 0.004486494767484458 -0.04212299439943243 -0.03264261977732932 0.09346324145116293 0.059159784737169076 -0.02189670812727101 -0.007561187422134356 0.038255369601437245 0.024014259275705073 -0.029235242190348093 -0.012280979435560314 0.0276184387386278 0.01276493644757829 -0.056234844739195364 -0.04068565578116063; 0.004486493947602797 -0.010006435105274674 -0.010831522194109528 -0.013933323510621997 0.059159784757594355 0.03744404394686794 -0.014119474114086638 -0.00490359506067532 0.024014259275705073 0.015074646085856201 -0.018352001573282675 -0.007709184423633915 0.012764936261665027 0.005142910755279294 -0.03180653565704714 -0.02407205427958841; -0.04212299488121628 -0.010831523119193389 0.041203264273983756 0.028731742931381858 -0.0218967061719174 -0.014119472861164016 -0.021117009463502447 -0.010121793911433849 -0.029235242190348093 -0.018352001573282675 0.022341967950087107 0.009385278182389144 -0.05623484614455744 -0.03180653668694516 0.06982097063502593 0.042369623822553895; -0.032642620569597956 -0.013933324280692464 0.028731743372867102 0.01686811409449618 -0.007561186520363213 -0.004903594483251425 -0.010121793952609958 -0.004777458937143693 -0.012280979435560314 -0.007709184423633915 0.009385278182389144 0.003942538881534688 -0.04068565704440573 -0.02407205513979285 0.042369624317095525 0.023276033404581897; 0.02865266515300746 -0.0015043458158730987 -0.0668546421287193 -0.05657731063379868 0.06747578507343358 0.03154000452326787 -0.10163212597069286 -0.08319810466268614 0.0276184387386278 0.012764936261665027 -0.05623484614455744 -0.04068565704440573 0.01993916598501361 0.005914807343982406 -0.06595971763033172 -0.05234500027696871; -0.0015043462202460841 -0.003413852320466667 0.013261580222624721 0.009479461803078005 0.03154000499124446 0.01277455744466305 -0.06826988455541702 -0.054932942344648426 0.01276493644757829 0.005142910755279294 -0.03180653668694516 -0.02407205513979285 0.005914807343982406 0.0017546056935234748 -0.01956646389577304 -0.015527771620336993; -0.06685464101272384 0.013261579227509238 0.12876482598366956 0.1138377921610535 -0.10163212111738537 -0.06826988156902396 -0.0659931177562143 -0.043956470091493345 -0.056234844739195364 -0.03180653565704714 0.06982097063502593 0.042369624317095525 -0.06595971763033172 -0.01956646389577304 0.21819839147744113 0.173160201384404; -0.05657730989307746 0.009479460973031458 0.11383779262999263 0.099586916927369 -0.08319810092528214 -0.05493294001185737 -0.04395646974962552 -0.028205329715387483 -0.04068565578116063 -0.02407205427958841 0.042369623822553895 0.023276033404581897 -0.05234500027696871 -0.015527771620336993 0.173160201384404 0.13741834648554363]

16×16 Matrix{Float64}:
  0.0411742   -0.0165374   -0.055935   …  -0.0668546  -0.0565773
 -0.0165374    0.00664226   0.0224661      0.0132616   0.00947946
 -0.055935     0.0224661    0.0759877      0.128765    0.113838
 -0.054512     0.0218945    0.0740545      0.113838    0.0995869
  0.0969632    0.0114691   -0.0515286     -0.101632   -0.0831981
  0.0114691   -0.0248551   -0.0477911  …  -0.0682699  -0.0549329
 -0.0515286   -0.0477911   -0.0389443     -0.0659931  -0.0439565
 -0.0564671   -0.0440652   -0.0294638     -0.0439565  -0.0282053
  0.0396879    0.00448649  -0.042123      -0.0562348  -0.0406857
  0.00448649  -0.0100064   -0.0108315     -0.0318065  -0.0240721
 -0.042123    -0.0108315    0.0412033  …   0.069821    0.0423696
 -0.0326426   -0.0139333    0.0287317      0.0423696   0.023276
  0.0286527   -0.00150435  -0.0668546     -0.0659597  -0.052345
 -0.00150435  -0.00341385   0.0132616     -0.0195665  -0.0155278
 -0.0668546    0.0132616    0.128765       0.218198    0.17316
 -0.05

In [5]:
using LinearAlgebra

In [12]:
dot(ppt, wit)

-1.0094654603054476e-6

In [35]:
findmin(dot.(forms, Ref(ppt)))

(23.87739518220789, 131)

In [38]:
using Ket
using MosekTools

In [ ]:
r, w = entanglement_robustness(ppt, [4, 4], 3; solver=Mosek.Optimizer)